In [1]:
from scipy.optimize import linprog

In [2]:
delay_probability=0.82
budget=20000
options={
    "Air Freight":{
        "cost":15000,
        "delay_days":3
    },
    "Secondary Supplier":{
        "cost":16500,
        "delay_days":5
    },
    "Delay Product Launch":{
        "cost":4000,
        "delay_days":14
    }
}

In [3]:
for action, values in options.items():
    score = values["cost"] + (values["delay_days"] * 1000)
    
    print(action, "→", score)

Air Freight → 18000
Secondary Supplier → 21500
Delay Product Launch → 18000


In [4]:
feasible_options = {
    action: values
    for action, values in options.items()
    if values["cost"] <= budget
}

feasible_options

{'Air Freight': {'cost': 15000, 'delay_days': 3},
 'Secondary Supplier': {'cost': 16500, 'delay_days': 5},
 'Delay Product Launch': {'cost': 4000, 'delay_days': 14}}

In [5]:
best_action = min(
    feasible_options,
    key=lambda action:
        feasible_options[action]["cost"]
        + feasible_options[action]["delay_days"] * 1000
)

best_action

'Air Freight'

In [6]:
import numpy as np
import pandas as pd

In [7]:
shipment={
    "shipment_id": "SHP01024",
    "order_quantity": 1200,
    "inventory_level": 800,
    "supplier_capacity": 5000,
    "shipping_cost": 12000,
    "delay_probability": 0.82
}

In [8]:
shipment

{'shipment_id': 'SHP01024',
 'order_quantity': 1200,
 'inventory_level': 800,
 'supplier_capacity': 5000,
 'shipping_cost': 12000,
 'delay_probability': 0.82}

In [9]:
actions=pd.DataFrame({
    "action":[
        "Air Freight",
        "Secondary supplier",
        "Delay Product Launch"
    ],
    "cost":[
        15000,
        16500,
        4000
    ],
    "delay_days":[
        3,5,14
    ],
    "capacity":[
        2000,
        3000,
        1200
    ]
})

In [10]:
actions

,action,cost,delay_days,capacity
0,Air Freight,15000,3,2000
1,Secondary supplier,16500,5,3000
2,Delay Product Launch,4000,14,1200


In [11]:
delay_penalty=1000
actions["objective_cost"]=(
    actions["cost"]+
    actions["delay_days"]*delay_penalty
)

In [12]:
actions

,action,cost,delay_days,capacity,objective_cost
0,Air Freight,15000,3,2000,18000
1,Secondary supplier,16500,5,3000,21500
2,Delay Product Launch,4000,14,1200,18000


In [13]:
c=actions["objective_cost"].values

In [14]:
c

array([18000, 21500, 18000])

In [15]:
A_eq = np.array([
    [1, 1, 1]
])

b_eq = np.array([1])

In [16]:
budget=20000

In [17]:
A_ub = np.array([
    actions["cost"].values
])

b_ub = np.array([
    budget
])

In [18]:
bounds = [
    (0, 1),
    (0, 1),
    (0, 1)
]

In [19]:
result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

In [20]:
result.success

True

In [21]:
result.x

array([1., 0., 0.])

In [22]:
selected_index = np.argmax(result.x)

selected_action = actions.iloc[selected_index]["action"]

selected_action

'Air Freight'

In [23]:
print("Recommended Action:", selected_action)
print(
    "Estimated Cost: $",
    actions.iloc[selected_index]["cost"]
)
print(
    "Expected Delay:",
    actions.iloc[selected_index]["delay_days"],
    "days"
)

Recommended Action: Air Freight
Estimated Cost: $ 15000
Expected Delay: 3 days


In [24]:
shipment = {
    "order_quantity": 1200,
    "inventory_level": 800,
    "supplier_capacity": 5000,
    "delay_probability": 0.82
}

budget = 20000
max_acceptable_delay = 7

In [25]:
actions = pd.DataFrame({
    "action": [
        "Air Freight",
        "Secondary Supplier",
        "Delay Product Launch"
    ],
    
    "cost": [
        15000,
        16500,
        4000
    ],
    
    "delay_days": [
        3,
        5,
        14
    ],
    
    "capacity": [
        2000,
        3000,
        1200
    ],
    
    "risk_reduction": [
        0.75,
        0.55,
        0.10
    ]
})

actions

,action,cost,delay_days,capacity,risk_reduction
0,Air Freight,15000,3,2000,0.75
1,Secondary Supplier,16500,5,3000,0.55
2,Delay Product Launch,4000,14,1200,0.10


In [26]:
actions["remaining_risk"] = (
    shipment["delay_probability"]
    * (1 - actions["risk_reduction"])
)

actions

,action,cost,delay_days,capacity,risk_reduction,remaining_risk
0,Air Freight,15000,3,2000,0.75,0.205
1,Secondary Supplier,16500,5,3000,0.55,0.369
2,Delay Product Launch,4000,14,1200,0.10,0.738


In [27]:
actions["budget_feasible"] = (
    actions["cost"] <= budget
)

In [28]:
actions["delay_feasible"] = (
    actions["delay_days"] <= max_acceptable_delay
)

In [29]:
actions["capacity_feasible"] = (
    actions["capacity"] >= shipment["order_quantity"]
)

In [30]:
actions[
    [
        "action",
        "cost",
        "delay_days",
        "remaining_risk",
        "budget_feasible",
        "delay_feasible",
        "capacity_feasible"
    ]
]

,action,cost,delay_days,remaining_risk,budget_feasible,delay_feasible,capacity_feasible
0,Air Freight,15000,3,0.205,True,True,True
1,Secondary Supplier,16500,5,0.369,True,True,True
2,Delay Product Launch,4000,14,0.738,True,False,True


In [31]:
feasible_actions = actions[
    actions["budget_feasible"]
    & actions["delay_feasible"]
    & actions["capacity_feasible"]
].copy()

feasible_actions

,action,cost,delay_days,capacity,risk_reduction,remaining_risk,budget_feasible,delay_feasible,capacity_feasible
0,Air Freight,15000,3,2000,0.75,0.205,True,True,True
1,Secondary Supplier,16500,5,3000,0.55,0.369,True,True,True


In [32]:
cost_weight = 0.4
risk_weight = 0.6

feasible_actions["score"] = (
    cost_weight
    * (feasible_actions["cost"] / budget)
    +
    risk_weight
    * feasible_actions["remaining_risk"]
)

In [33]:
recommendations = feasible_actions.sort_values(
    "score"
).reset_index(drop=True)

recommendations[
    [
        "action",
        "cost",
        "delay_days",
        "remaining_risk",
        "score"
    ]
]

,action,cost,delay_days,remaining_risk,score
0,Air Freight,15000,3,0.205,0.4230
1,Secondary Supplier,16500,5,0.369,0.5514


In [34]:
best_action = recommendations.iloc[0]

print("Recommended Action:", best_action["action"])
print("Cost: $", best_action["cost"])
print("Expected Delay:", best_action["delay_days"], "days")
print(
    "Remaining Delay Risk:",
    round(best_action["remaining_risk"] * 100, 2),
    "%"
)

Recommended Action: Air Freight
Cost: $ 15000
Expected Delay: 3 days
Remaining Delay Risk: 20.5 %


In [35]:
recommendations[
    ["action", "cost", "delay_days", "remaining_risk", "score"]
]

,action,cost,delay_days,remaining_risk,score
0,Air Freight,15000,3,0.205,0.4230
1,Secondary Supplier,16500,5,0.369,0.5514


In [36]:
import pandas as pd

In [37]:
data=pd.read_csv("../data/raw/supply_chain_data.csv")

In [38]:
data.head()

,shipment_id,supplier,product,distance_km,order_quantity,supplier_reliability,historical_delay_rate,lead_time_days,inventory_level,supplier_capacity,shipping_cost,weather_risk,demand_forecast,delay
0,SHP00001,Supplier_C,Batteries,805,972,0.77,0.33,14,1208,5552,3109.13,0.27,660,1
1,SHP00002,Supplier_D,Sensors,1120,601,0.93,0.05,5,287,4501,10443.46,0.56,2301,0
2,SHP00003,Supplier_A,Processors,2163,242,0.87,0.13,16,1545,3077,5077.64,0.39,1638,1
3,SHP00004,Supplier_C,Batteries,1546,1712,0.77,0.18,7,1099,9526,17613.17,0.17,2967,1
4,SHP00005,Supplier_C,Microchips,3764,338,0.84,0.24,2,236,5755,3879.20,0.07,2531,1


In [39]:
shipment=data.iloc[0]

In [40]:
shipment

shipment_id                SHP00001
supplier                 Supplier_C
product                   Batteries
distance_km                     805
order_quantity                  972
supplier_reliability           0.77
historical_delay_rate          0.33
lead_time_days                   14
inventory_level                1208
supplier_capacity              5552
shipping_cost               3109.13
weather_risk                   0.27
demand_forecast                 660
delay                             1
Name: 0, dtype: object

In [41]:
import joblib

In [42]:
preprocessor=joblib.load("../ml/preprocessor.pkl")

In [43]:
model = joblib.load(
    "../ml/xgboost_delay_model.pkl"
)

In [44]:
shipment_features = shipment.drop(
    labels=["shipment_id", "delay"]
).to_frame().T

In [45]:
shipment_processed = preprocessor.transform(
    shipment_features
)

In [46]:
delay_probability = model.predict_proba(
    shipment_processed
)[0, 1]

print(
    "Delay Probability:",
    round(delay_probability * 100, 2),
    "%"
)

Delay Probability: 68.27 %


In [49]:
actions = pd.DataFrame({
    "action": [
        "Air Freight",
        "Secondary Supplier",
        "Delay Product Launch"
    ],
    
    "cost": [
        shipment_info["shipping_cost"] * 1.25,
        shipment_info["shipping_cost"] * 1.40,
        shipment_info["shipping_cost"] * 0.35
    ],
    
    "delay_days": [
        3,
        5,
        14
    ],
    
    "capacity": [
        2000,
        3000,
        shipment_info["order_quantity"]
    ],
    
    "risk_reduction": [
        0.75,
        0.55,
        0.10
    ]
})

actions

,action,cost,delay_days,capacity,risk_reduction
0,Air Freight,3886.4125,3,2000,0.75
1,Secondary Supplier,4352.7820,5,3000,0.55
2,Delay Product Launch,1088.1955,14,972,0.10


In [50]:
actions["remaining_risk"] = (
    delay_probability
    * (1 - actions["risk_reduction"])
)

actions

,action,cost,delay_days,capacity,risk_reduction,remaining_risk
0,Air Freight,3886.4125,3,2000,0.75,0.170685
1,Secondary Supplier,4352.7820,5,3000,0.55,0.307233
2,Delay Product Launch,1088.1955,14,972,0.10,0.614465


In [51]:
budget = 20000
max_acceptable_delay = 7

In [52]:
actions["budget_feasible"] = (
    actions["cost"] <= budget
)

In [53]:
actions["delay_feasible"] = (
    actions["delay_days"] <= max_acceptable_delay
)

In [54]:
actions["capacity_feasible"] = (
    actions["capacity"] >= shipment_info["order_quantity"]
)

In [55]:
feasible_actions = actions[
    actions["budget_feasible"]
    & actions["delay_feasible"]
    & actions["capacity_feasible"]
].copy()

feasible_actions

,action,cost,delay_days,capacity,risk_reduction,remaining_risk,budget_feasible,delay_feasible,capacity_feasible
0,Air Freight,3886.4125,3,2000,0.75,0.170685,True,True,True
1,Secondary Supplier,4352.7820,5,3000,0.55,0.307233,True,True,True


In [56]:
cost_weight = 0.4
risk_weight = 0.6

feasible_actions["score"] = (
    cost_weight
    * (feasible_actions["cost"] / budget)
    +
    risk_weight
    * feasible_actions["remaining_risk"]
)

In [57]:
recommendations = (
    feasible_actions
    .sort_values("score")
    .reset_index(drop=True)
)

In [58]:
recommendations[
    [
        "action",
        "cost",
        "delay_days",
        "remaining_risk",
        "score"
    ]
]

,action,cost,delay_days,remaining_risk,score
0,Air Freight,3886.4125,3,0.170685,0.180139
1,Secondary Supplier,4352.7820,5,0.307233,0.271395


In [59]:
best_action = recommendations.iloc[0]

print("Recommended Action:", best_action["action"])
print("Estimated Cost: $", round(best_action["cost"], 2))
print("Expected Delay:", best_action["delay_days"], "days")
print(
    "Remaining Delay Risk:",
    round(best_action["remaining_risk"] * 100, 2),
    "%"
)

Recommended Action: Air Freight
Estimated Cost: $ 3886.41
Expected Delay: 3 days
Remaining Delay Risk: 17.07 %


In [60]:
recommendations[
    ["action", "cost", "delay_days", "remaining_risk", "score"]
]

,action,cost,delay_days,remaining_risk,score
0,Air Freight,3886.4125,3,0.170685,0.180139
1,Secondary Supplier,4352.7820,5,0.307233,0.271395
